# 01 · Frame-level baseline (SigLIP, 1 fps)

**Kaggle settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Use **Add Input** to attach the dataset `charades-sta-vidsearch`.

Plan:
1. Unpack the videos.
2. Embed every frame at 1 fps with SigLIP (GPU, once). Cache the result.
3. Embed every query (GPU, once). Cache the result.
4. Cosine similarity between frames and query gives a relevance curve over time.
5. Turn the curve into segments in two ways:
   - **(i)** threshold and merge consecutive frames (the baseline described in the proposal)
   - **(ii)** multi-scale sliding windows with contrast scoring
6. Tune each on **val** (CPU, fast), report once on **test**, and log everything.

**Save GPU quota:** run interactively with `DEV = True` (30 videos, a few minutes). Then set `DEV = False` and use **Save & Run All (Commit)** for the full run, which takes roughly 20–30 GPU minutes.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/zankhana46/AI-Video-Investigator.git"
!rm -rf /tmp/repo && git clone -q $REPO_URL /tmp/repo
import sys
sys.path.insert(0, "/tmp/repo")
!python /tmp/repo/tests/test_basics.py && python /tmp/repo/tests/test_proposals.py

## 2. Unpack the videos from the dataset

In [ ]:
from pathlib import Path
tar_path = next(Path("/kaggle/input").rglob("videos.tar"))   # works whatever the mount path is
DATA = tar_path.parent
!tar -xf "$tar_path" -C /tmp
VID_DIR = Path("/tmp/videos")
print("dataset:", DATA, "| videos unpacked:", len(list(VID_DIR.glob("*.mp4"))))   # expect 1634

## 3. Settings and annotations

In [ ]:
import shutil, json
from vidsearch.data import load_jsonl

DEV   = True                                  # True = 30 videos (interactive check); False = full run
MODEL = "google/siglip-base-patch16-224"
TAG   = "siglip_base"
FPS   = 1.0

WORK = Path("/kaggle/working"); FEAT_DIR = WORK / "features"; RES_DIR = WORK / "results"
for d in (FEAT_DIR, RES_DIR): d.mkdir(parents=True, exist_ok=True)
if (DATA / "results" / "results.csv").exists():                  # continue the experiment log
    shutil.copy(DATA / "results" / "results.csv", RES_DIR / "results.csv")

val  = load_jsonl(DATA / "annotations" / "val.jsonl")
test = load_jsonl(DATA / "annotations" / "test.jsonl")
vids = sorted({r["vid"] for r in val + test})
if DEV:
    keep = set(vids[:15] + sorted({r["vid"] for r in test})[:15])
    vids = sorted(keep)
    val  = [r for r in val  if r["vid"] in keep]
    test = [r for r in test if r["vid"] in keep]
print(f"videos {len(vids)} | val queries {len(val)} | test queries {len(test)} | DEV={DEV}")

## 4. Load SigLIP on the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
from vidsearch.features import (load_model, extract_video_features, encode_texts,
                                save_video_features, save_text_features)
model, processor = load_model(MODEL)
print("loaded", MODEL, "| dtype", model.dtype)

## 5. Extract frame features (the only heavy GPU step)

CPU threads decode the next videos while the GPU embeds the current one. The printed stats are your **indexing-efficiency** numbers for the report.

In [ ]:
video_feats, idx_stats = extract_video_features(vids, VID_DIR, model, processor, fps=FPS, batch_size=64)
save_video_features(video_feats, FEAT_DIR / f"{TAG}_video_{FPS:g}fps.npz",
                    meta={"model": MODEL, "fps": FPS, **idx_stats})
print(f"{idx_stats['n_videos']} videos, {idx_stats['n_frames']} frames, "
      f"{idx_stats['video_minutes']:.0f} video-minutes in {idx_stats['seconds']/60:.1f} min")
print(f"indexing speed: {idx_stats['sec_per_video_minute']:.2f} s per video-minute | "
      f"peak GPU memory {idx_stats['peak_gpu_mem_gb']:.2f} GB | failed: {idx_stats['failed']}")
print("feature shape example:", next(iter(video_feats.values())).shape)

## 6. Encode all queries

In [ ]:
from vidsearch.utils import timer
recs = val + test
with timer("encode queries"):
    E = encode_texts(model, processor, [r["query"] for r in recs])
save_text_features([r["qid"] for r in recs], E, FEAT_DIR / f"{TAG}_text.npz")
text_feats = dict(zip([r["qid"] for r in recs], E))
print("text emb shape:", E.shape)

## 7. Is there signal? (sanity check before any tuning)

For each val query, compare the mean similarity *inside* the ground-truth moment with the mean *outside* it. If SigLIP carries useful signal, "inside > outside" should hold well above 50% of the time.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from vidsearch.search import score_curves
val  = [r for r in val  if r["vid"] in video_feats]       # drop queries whose video failed
test = [r for r in test if r["vid"] in video_feats]
curves_val, curves_test = score_curves(val, video_feats, text_feats), score_curves(test, video_feats, text_feats)

def inside_mask(r, n):
    t = (np.arange(n) + 0.5) / FPS
    return (t >= r["start"]) & (t <= r["end"])

wins = []
for r in val:
    c = curves_val[r["qid"]]; m = inside_mask(r, len(c))
    if m.any() and (~m).any():
        wins.append(c[m].mean() > c[~m].mean())
print(f"inside > outside for {np.mean(wins):.1%} of val queries (50% = no signal)")

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for ax, r in zip(axes.flat, test[:4]):
    c = curves_test[r["qid"]]; t = (np.arange(len(c)) + 0.5) / FPS
    ax.plot(t, c); ax.axvspan(r["start"], r["end"], color="green", alpha=.25, label="GT")
    ax.set_title(r["query"][:60], fontsize=10); ax.set_xlabel("seconds"); ax.legend()
plt.tight_layout(); plt.savefig(RES_DIR / "frame_curves_examples.png", dpi=120); plt.show()

## 8. Baseline (i): threshold + merge consecutive frames, tuned on val

In [ ]:
import pandas as pd
from vidsearch.search import predict, grid_search
from vidsearch.metrics import evaluate
from vidsearch.utils import log_result

grid_thr = {"thr": [0.0, 0.25, 0.5, 0.75, 1.0, 1.5], "smooth_k": [1, 3, 5]}
rows = grid_search(val, curves_val, grid_thr, method="threshold", fps=FPS)
display(pd.DataFrame(rows).head(5).round(1))
best_thr = {k: rows[0][k] for k in grid_thr}
print("best on val:", best_thr)

sum_thr, pq_thr = evaluate(predict(test, curves_test, "threshold", fps=FPS, **best_thr), test)
print("TEST:", {k: round(v, 1) for k, v in sum_thr.items()})
if not DEV:
    log_result(RES_DIR / "results.csv", f"frame_threshold_{TAG}", "test", sum_thr, best_thr)

## 9. Baseline (ii): multi-scale sliding windows with contrast, tuned on val

In [ ]:
grid_win = {
    "smooth_k":  [1, 3, 5],
    "ctx":       [0.0, 0.25, 0.5, 1.0],
    "lengths_s": [(2, 4, 8, 16, 32), (4, 8, 12, 16, 24), (3, 5, 8, 12, 16, 24, 32)],
    "nms_thr":   [0.3, 0.5],
}
with timer("grid search (CPU)"):
    rows = grid_search(val, curves_val, grid_win, method="windows", fps=FPS)
display(pd.DataFrame(rows).head(8).round(1))
best_win = {k: rows[0][k] for k in grid_win}
print("best on val:", best_win)

preds_win = predict(test, curves_test, "windows", fps=FPS, **best_win)
sum_win, pq_win = evaluate(preds_win, test)
print("TEST:", {k: round(v, 1) for k, v in sum_win.items()})
if not DEV:
    log_result(RES_DIR / "results.csv", f"frame_windows_{TAG}", "test", sum_win,
               {**best_win, "lengths_s": list(best_win["lengths_s"])})

## 10. Query latency (search time per query, excluding the one-time indexing)

In [ ]:
import time
lat = []
for r in test[:200]:
    t0 = time.perf_counter()
    q = encode_texts(model, processor, [r["query"]])[0]                        # GPU text encoding
    s = video_feats[r["vid"]].astype(np.float32) @ q                           # score every frame
    predict([r], {r["qid"]: s}, "windows", fps=FPS, **best_win)                # segments
    lat.append(time.perf_counter() - t0)
eff = {"model": MODEL, "fps": FPS, "query_latency_ms_mean": 1000 * np.mean(lat),
       "query_latency_ms_p95": 1000 * np.percentile(lat, 95),
       **{k: v for k, v in idx_stats.items() if k != "failed"}}
(RES_DIR / f"efficiency_frame_{TAG}.json").write_text(json.dumps(eff, indent=1))
print({k: round(v, 2) if isinstance(v, float) else v for k, v in eff.items()})

## 11. Results so far and per-query outputs (needed for the challenging-case analysis)

In [ ]:
(RES_DIR / f"per_query_frame_windows_{TAG}.json").write_text(json.dumps(pq_win))
(RES_DIR / f"per_query_frame_threshold_{TAG}.json").write_text(json.dumps(pq_thr))
(RES_DIR / f"preds_frame_windows_{TAG}.json").write_text(json.dumps(preds_win))
if (RES_DIR / "results.csv").exists():
    display(pd.read_csv(RES_DIR / "results.csv")[["method", "R1@0.5", "R1@0.7", "R5@0.5", "R5@0.7", "mIoU"]])
!ls -lh /kaggle/working/features /kaggle/working/results

## 12. Save

1. Set `DEV = False` in cell 3.
2. Click **Save Version → Save & Run All (Commit)**.
3. When it finishes, check the **Output** tab: you should see `features/` (2 `.npz` files and 1 `.json`) and `results/`.
4. Create a dataset from the output called **`vidsearch-features`**, or attach this notebook's output directly with **Add Input → Notebooks**. Notebooks 02 and 03 load features from there, so the GPU never re-embeds frames.

**What to look for:** test R1@0.5 for (ii) should clearly beat the prior's 13.8, and (ii) should beat (i). Send me the results table.